In [94]:
import numpy as np
import tensorflow as tf
import pandas as pd
import re
import nltk

In [111]:
df=pd.read_csv('Restaurant_Reviews.tsv', sep='\t')
df

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1
...,...,...
995,I think food should have flavor and texture an...,0
996,Appetite instantly gone.,0
997,Overall I was not impressed and would not go b...,0
998,"The whole experience was underwhelming, and I ...",0


In [96]:
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Embedding


In [114]:
for i in range(len(df)):
    review = re.sub('[^a-zA-Z]', ' ', df.loc[i, 'Review'])
    df.loc[i, 'Review'] = review.lower()
    df.loc[i, 'Review'] = review.strip()
df

,Review,Liked
0,Wow Loved this place,1
1,Crust is not good,0
2,Not tasty and the texture was just nasty,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1
...,...,...
995,I think food should have flavor and texture an...,0
996,Appetite instantly gone,0
997,Overall I was not impressed and would not go back,0
998,The whole experience was underwhelming and I ...,0


In [115]:
#Giving index numbers of the words
vocab_size=10000
encoded_reviews=[one_hot(d, vocab_size) for d in df['Review']]
encoded_reviews

[[5667, 7631, 7703, 1663],
 [7276, 4070, 4699, 5109],
 [4699, 7630, 3193, 1635, 1983, 7407, 3013, 9994],
 [2409,
  839,
  1690,
  1635,
  4887,
  9162,
  701,
  687,
  9423,
  3463,
  63,
  8451,
  3193,
  7631,
  5577],
 [1635, 4516, 9150, 1635, 5603, 7407, 8375, 3193, 4617, 552, 1635, 6881],
 [3167, 7086, 6490, 6009, 5154, 3193, 7086, 1317, 8179, 2910, 1599],
 [648, 5577, 648, 6912, 7154, 2838, 6763],
 [1635,
  2617,
  552,
  7439,
  5747,
  3193,
  5467,
  5520,
  2262,
  115,
  8688,
  652,
  2202,
  3064,
  2210,
  6914,
  9322,
  1569,
  3825,
  1783,
  2109,
  9777],
 [1635, 1318, 552, 8375, 957],
 [2109, 8375, 9117],
 [5965, 7407, 9518, 7841],
 [974, 4699, 6874, 3355],
 [1635,
  6222,
  8688,
  8394,
  5933,
  567,
  4617,
  8880,
  9150,
  567,
  7086,
  8688,
  8639,
  3178,
  5577,
  3711,
  1003,
  3064,
  1569,
  2254,
  6094],
 [7086, 3441, 1635, 9504, 7976, 7922, 6408, 624, 4639, 4514],
 [7086, 7407, 7944, 2510, 7086, 7407, 3435, 6362, 2838, 7407, 511, 5561],
 [7086, 740

In [116]:
max_length=max([len(d) for d in encoded_reviews])
padded_reviews=pad_sequences(encoded_reviews, maxlen=max_length, padding='post')
padded_reviews

array([[5667, 7631, 7703, ...,    0,    0,    0],
       [7276, 4070, 4699, ...,    0,    0,    0],
       [4699, 7630, 3193, ...,    0,    0,    0],
       ...,
       [8121, 7086, 7407, ...,    0,    0,    0],
       [1635, 1109, 7393, ...,    0,    0,    0],
       [9136, 5359,  961, ...,    0,    0,    0]],
      shape=(1000, 32), dtype=int32)

In [81]:
help(pad_sequences)

Help on function pad_sequences in module keras.src.utils.sequence_utils:

pad_sequences(sequences, maxlen=None, dtype='int32', padding='pre', truncating='pre', value=0.0)
    Pads sequences to the same length.
    
    This function transforms a list (of length `num_samples`)
    of sequences (lists of integers)
    into a 2D NumPy array of shape `(num_samples, num_timesteps)`.
    `num_timesteps` is either the `maxlen` argument if provided,
    or the length of the longest sequence in the list.
    
    Sequences that are shorter than `num_timesteps`
    are padded with `value` until they are `num_timesteps` long.
    
    Sequences longer than `num_timesteps` are truncated
    so that they fit the desired length.
    
    The position where padding or truncation happens is determined by
    the arguments `padding` and `truncating`, respectively.
    Pre-padding or removing values from the beginning of the sequence is the
    default.
    
    >>> sequence = [[1], [2, 3], [4, 5, 6]]
 

In [117]:
embedded_vector_size=5
model=Sequential()
model.add(Embedding(vocab_size, embedded_vector_size, input_length=max_length, name='embedding' ))
model.add(Flatten())
model.add(Dense(1, activation='sigmoid'))

d:\git_prac\NLP\.venv311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [118]:
X=padded_reviews
y=df.iloc[:, 1]
X

array([[5667, 7631, 7703, ...,    0,    0,    0],
       [7276, 4070, 4699, ...,    0,    0,    0],
       [4699, 7630, 3193, ...,    0,    0,    0],
       ...,
       [8121, 7086, 7407, ...,    0,    0,    0],
       [1635, 1109, 7393, ...,    0,    0,    0],
       [9136, 5359,  961, ...,    0,    0,    0]],
      shape=(1000, 32), dtype=int32)

In [119]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X, y, epochs=10, batch_size=32)


Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5270 - loss: 0.6928   
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6610 - loss: 0.6870 
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7380 - loss: 0.6805 
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7600 - loss: 0.6718 
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7990 - loss: 0.6595 
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8270 - loss: 0.6433 
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 999us/step - accuracy: 0.8620 - loss: 0.6233
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8830 - loss: 0.5991 
Epoch 9/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8900 - loss: 0.5717 
Epoch 10/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8990 - loss: 0.5414 


In [120]:
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 32, 5)          │        50,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_7 (Flatten)             │ (None, 160)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           161 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 150,485 (587.84 KB)

 Trainable params: 50,161 (195.94 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 100,324 (391.89 KB)

In [121]:
model.evaluate(X,y)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 975us/step - accuracy: 0.9090 - loss: 0.5212


[0.5212224721908569, 0.9089999794960022]

In [122]:
model.get_layer('embedding').get_weights()[0]

array([[-0.01145842, -0.04016282,  0.03682453, -0.03725497,  0.04294382],
       [ 0.01266683,  0.03472258, -0.0482116 ,  0.01416024, -0.01070535],
       [-0.00445373, -0.03090613, -0.02783087, -0.01920329,  0.02171738],
       ...,
       [ 0.01828439,  0.00140122,  0.01590181,  0.0288333 , -0.0232108 ],
       [ 0.01911911,  0.01584153, -0.03306593,  0.09475888, -0.12033331],
       [ 0.04005729, -0.00916981, -0.02863294,  0.04730011, -0.00490508]],
      shape=(10000, 5), dtype=float32)

In [123]:
(padded_reviews)[0]

array([5667, 7631, 7703, 1663,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
      dtype=int32)

In [124]:
print(model.predict(padded_reviews)[0])

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
[0.6335859]
